<a href="https://colab.research.google.com/github/ajinfajrian/data-science-2026/blob/master/Pertemuan3_Fajrian_Ichlasul_240401020100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# Memuat dataset asli yang sudah kamu upload ke folder local Google Colab
df = pd.read_csv('https://files.ajinf.my.id/housing_dirty.csv')

# Menampilkan dimensi awal dan mengecek kolom yang bolong
print(f"Dimensi Awal Dataset: {df.shape} (Baris, Kolom)\n")
print("--- Jumlah Missing Values Awal Per Kolom ---")
print(df.isnull().sum())

Dimensi Awal Dataset: (130, 7) (Baris, Kolom)

--- Jumlah Missing Values Awal Per Kolom ---
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [6]:
# 1. Menghapus baris duplikat jika ada baris yang sama persis
df.drop_duplicates(inplace=True)

# 2. Standardisasi Nama Kota agar seragam sesuai data aslimu
pemetaan_kota = {
    'jogja': 'Yogyakarta',
    'YGY': 'Yogyakarta',
    'sby': 'Surabaya',
    'dpk': 'Depok',
    'mdn': 'Medan',
    'jakarta': 'Jakarta',
    'makassar': 'Makassar',
    'semarang': 'Semarang',
    'medan': 'Medan'
}
df['kota'] = df['kota'].replace(pemetaan_kota)

# 3. Standardisasi kolom kondisi dan kota menjadi huruf kecil semua agar konsisten
df['kota'] = df['kota'].str.lower()
df['kondisi'] = df['kondisi'].str.lower()

print("--- Hasil Standardisasi Teks (5 Baris Pertama) ---")
print(df[['kota', 'kondisi']].head())

--- Hasil Standardisasi Teks (5 Baris Pertama) ---
         kota kondisi
0  yogyakarta    baik
1       medan   bagus
2       depok    baik
3  yogyakarta    baik
4       medan  sedang


In [7]:
# Imputasi kolom numerik kontinu (luas_m2, harga_juta) menggunakan Median (Aman dari outlier)
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Imputasi kolom tahun_bangun menggunakan Median
df['tahun_bangun'] = df['tahun_bangun'].fillna(df['tahun_bangun'].median())

# Imputasi kolom diskret/kategorikal (kamar, kota, kondisi) menggunakan Modus
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
df['kota'] = df['kota'].fillna(df['kota'].mode()[0])
df['kondisi'] = df['kondisi'].fillna(df['kondisi'].mode()[0])

print("--- Jumlah Missing Values Setelah Imputasi ---")
print(df.isnull().sum())

--- Jumlah Missing Values Setelah Imputasi ---
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [8]:
# 1. Koreksi manual untuk tahun bangun yang typo ekstrem (9999) sebelum masuk ke IQR
df['tahun_bangun'] = df['tahun_bangun'].replace(9999, df['tahun_bangun'].median())

# 2. Menerapkan IQR Clipping untuk kolom luas_m2 dan harga_juta
for col in ['luas_m2', 'harga_juta', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    # Batasi nilai ekstrem agar sejajar dengan batas atas/bawah toleransi IQR
    df[col] = df[col].clip(lower_fence, upper_fence)

print("--- Ringkasan Statistik Setelah Outlier Diatasi ---")
print(df[['luas_m2', 'harga_juta', 'tahun_bangun']].describe().round(2))

--- Ringkasan Statistik Setelah Outlier Diatasi ---
       luas_m2  harga_juta  tahun_bangun
count   130.00      130.00        130.00
mean    188.27      686.49       2001.22
std      95.30      404.63         12.94
min     -50.00     -422.75       1961.62
25%     101.60      380.50       1991.25
50%     193.80      655.00       2002.00
75%     266.15      916.00       2011.00
max     512.97     1719.25       2040.62


In [9]:
# Validasi ketat menggunakan assert memastikan data 100% bersih sebelum diekspor
assert df.isnull().sum().sum() == 0, "Peringatan: Masih ada missing value!"
assert df.duplicated().sum() == 0, "Peringatan: Masih ada data duplikat!"

print(f"Validasi Sukses! Ukuran DataFrame Akhir: {df.shape}")

# Ekspor menjadi file CSV baru sesuai instruksi modul
df.to_csv('housing_clean.csv', index=False)
print("File 'housing_clean.csv' berhasil dibuat di penyimpanan lokal Colab!")

Validasi Sukses! Ukuran DataFrame Akhir: (130, 7)
File 'housing_clean.csv' berhasil dibuat di penyimpanan lokal Colab!
